<a href="https://colab.research.google.com/github/perezbrotonsluis/movie-tv-recommender/blob/main/notebooks/03_modelo_recomendador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#MODELO TF-IDF

##Introducción

Este notebook parte del dataset de películas y series ya preprocesado. El objetivo es construir un recomendador basado en contenido. Para ello, se empleará la técnica TF-IDF en los metadatos de los títulos y calcularemos la similitud coseno entre ellos. Esto nos permitirá identificar y sugerir títulos similares a partir de sus características.

## Carga del dataset

In [1]:
import pandas as pd

movie_tv_clean_url = "https://raw.githubusercontent.com/perezbrotonsluis/movie-tv-recommender/main/data/processed/movie_tv_clean.csv"
df = pd.read_csv(movie_tv_clean_url)

print("DataFrame shape:", df.shape)
print("\nAvailable columns:", df.columns.tolist())
print("\nDataFrame info (metadata):")
df.info()

DataFrame shape: (20263, 6)

Available columns: ['id', 'title', 'description', 'genres', 'names_combined', 'metadata']

DataFrame info (metadata):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20263 entries, 0 to 20262
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              20263 non-null  object
 1   title           20263 non-null  object
 2   description     20263 non-null  object
 3   genres          19945 non-null  object
 4   names_combined  18755 non-null  object
 5   metadata        20263 non-null  object
dtypes: object(6)
memory usage: 950.0+ KB


In [2]:
df_model = df[['id', 'title', 'metadata']]

##Vectorización con TF-IDF

Aquí, se inicializa el vectorizador TF-IDF definiendo sus parámetros clave:
- min_df
- max_df
- ngram_range.

Posteriormente, se ajusta el vectorizador al corpus de metadatos y el resultado es una matriz TF-IDF, que tiene dimensiones de n_títulos × n_features (número de títulos por número de características o términos únicos).

In [3]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(min_df=5, max_df=0.8, ngram_range=(1,2))

tfidf_matrix = tfidf_vectorizer.fit_transform(df_model['metadata'])

print("Shape of TF-IDF matrix:", tfidf_matrix.shape)
print("Number of features (unique terms):", len(tfidf_vectorizer.get_feature_names_out()))

print("\nSe guarda el modelo ...")
joblib.dump(tfidf_vectorizer, 'vectorizador_tfidf.joblib')

Shape of TF-IDF matrix: (20263, 48229)
Number of features (unique terms): 48229

Se guarda el modelo ...

Se guarda la matriz de similitud coseno ...


NameError: name 'cosine_sim_matrix' is not defined

##Análisis básico de la matriz TF-IDF (opcional pero recomendable)

En esta sección, lo que se muestra es:

- El tamaño de la matriz TF-IDF: sus dimensiones, es decir, n_títulos × n_features.
- El número de features: que representa el tamaño del vocabulario único identificado.
- La sparsity (dispersión): el porcentaje de ceros en la matriz.

In [4]:
print(f"Shape of TF-IDF matrix: {tfidf_matrix.shape}")
print(f"Number of features (vocabulary size): {tfidf_matrix.shape[1]}")

# Calculate sparsity
num_elements = tfidf_matrix.shape[0] * tfidf_matrix.shape[1]
num_non_zero = tfidf_matrix.nnz
sparsity = (1 - (num_non_zero / num_elements)) * 100

print(f"Sparsity of TF-IDF matrix: {sparsity:.2f}%")

Shape of TF-IDF matrix: (20263, 48229)
Number of features (vocabulary size): 48229
Sparsity of TF-IDF matrix: 99.87%


La matriz TF-IDF que hemos obtenido presenta una alta dispersión, superando el 99%. Esto es completamente normal y esperable en modelos basados en texto, ya que cada título individualmente solo utiliza una pequeña fracción del vocabulario total identificado, resultando en muchos ceros.

##Cálculo de la similitud entre títulos

Después de tener ya la matriz TF-IDF, el siguiente paso crucial es calcular la similitud entre los vectores de cada título. Para ello:

- Se calcula la similitud del coseno. Es una métrica que mide el ángulo entre dos vectores, indicando cuán similares son en dirección sin importar su magnitud.
- Se obtiene una matriz de similitud. El resultado es una matriz donde cada celda (i, j) contiene la similitud coseno entre el título i y el título j.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim_matrix = cosine_similarity(tfidf_matrix)

print("Shape of cosine similarity matrix:", cosine_sim_matrix.shape)

Shape of cosine similarity matrix: (20263, 20263)

Se guarda la matriz de similitud coseno ...


['cosine_sim_matrix.joblib']

##Creación del índice de títulos

Para que el sistema de recomendación funcione de manera ágil, es fundamental crear un índice de títulos. Este índice es una estructura que relaciona cada título de película con su índice numérico dentro del DataFrame.

Su principal utilidad es localizar rápidamente un título y
acceder a su vector de similitud. De esta manera se evitan búsquedas lentas y se consigue un recomendador mucho más eficiente.

In [8]:
title_to_index = pd.Series(df_model.index, index=df_model['title'])

print("Example of title to index mapping (first 5):")
print(title_to_index.head())

title_to_index.to_csv('title_to_index.csv')

Example of title to index mapping (first 5):
title
The Three Stooges              0
The General                    1
The Best Years of Our Lives    2
His Girl Friday                3
In a Lonely Place              4
dtype: int64


## Implementación de la función recomendadora

Se ha desarrollado una función para que al recibir una película o serie se pueda realizar todo el proceso de la recomencación y se devuelva un listado con el título recomendado y su correspondeinte puntaicón de similidtud

1. Recibe un título.
2. Se localiza su índice.
3. Se obtienen sus similitudes.
4. Se ordena de mayor a menor
5. Se Devuelve los top-N títulos más similares.


In [ ]:
def get_recommendations(title, cosine_sim_matrix, df_model, title_to_index, top_n=10):
    # Check if the title exists in our index
    if title not in title_to_index:
        return f"Title '{title}' not found in the dataset."

    # Get the index of the title that matches the input
    idx = title_to_index[title]

    # Get the pairwise similarity scores for all titles with that title
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Sort the titles based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the top_n most similar titles (excluding itself)
    sim_scores = sim_scores[1:top_n+1] # +1 because the first one is the title itself

    # Get the titles indices
    movie_indices = [i[0] for i in sim_scores]

    # Get the similarity scores
    similarity_scores = [i[1] for i in sim_scores]

    # Return the top_n most similar titles along with their similarity scores
    recommendations = df_model['title'].iloc[movie_indices]
    return pd.DataFrame({'Recommended Title': recommendations, 'Similarity Score': similarity_scores})

##Pruebas del recomendador y análisis

El objetivo de estas pruebas es verificar que el pipeline completo se ejecuta sin errores, que la función get_recommendations devuelve resultados y que estos tienen un sentido.

Para ello, se han elegido 3 títulos representativos del dataset: 'The Shawshank Redemption', 'Breaking Bad' y 'Game of Thrones'. Para cada uno, se ejecutará el recomendador y se analizaran las predicciones.

In [ ]:
# Test with a few representative titles
test_titles = ['The Shawshank Redemption', 'Breaking Bad', 'Game of Thrones']

for title in test_titles:
    print(f"\nRecommendations for '{title}':")
    recommendations = get_recommendations(title, cosine_sim_matrix, df_model, title_to_index, top_n=10)
    if isinstance(recommendations, str):
        print(recommendations)
    else:
        print(recommendations)



Recommendations for 'The Shawshank Redemption':
                 Recommended Title  Similarity Score
11892            A Christmas Story          0.127504
3657   Abducted: Fugitive for Love          0.107580
15856                         Riot          0.103852
4144       An American Ghost Story          0.101661
15335                    Wentworth          0.096753
1060      Where the Red Fern Grows          0.096614
15041                      Banyuki          0.096080
3217                  Greenfingers          0.094500
12011               The Green Mile          0.093398
11712               The Music Room          0.093334

Recommendations for 'Breaking Bad':
               Recommended Title  Similarity Score
15025                  Linewatch          0.166009
8381              The Road Ahead          0.125916
8556                  Alone Wolf          0.123378
15146                 Lilyhammer          0.120674
9184                      Mahaan          0.119191
14598             Strange

####The Shawshank Redemption

En este caso, el recomendador ofrece resultados mayoritariamente poco coherentes. Aunque el modelo identifica correctamente The Green Mile (comparten autor y temática carcelaria), el resto de las sugerencias se pierden en géneros opuestos como la comedia o el terror.

####Breaking Bad

Al analizar esta serie se puede ver como el modelo consigue algunos aciertos temáticos lógicos como "Escobar: Paradise Lost" o "Lilyhammer" al detectar palabras clave sobre crimen y narcotráfico. Sin embargo, también introduce mucho ruido con títulos totalmente irrelevantes.

####Game of Thrones

Aquí el modelo logra su puntuación de similitud más alta (0.29) con el documental "The Last Watch". Es un resultado esperado: al compartir nombres de actores y lugares, la coincidencia textual es directa. No obstante, en cuanto salimos de los nombres propios, el sistema confunde la fantasía épica con cualquier contenido que mencione "guerra" o "lucha", mezclando series de magia con películas de ciencia ficción o documentales bélicos.

####Conclución

Tras analizar estos tres casos, la conclusión es clara: el sistema funciona correctamente desde el punto de vista técnico, pero tiene limitaciones conceptuales importantes.

Al basarse puramente en TF-IDF, el recomendador encontrando secuelas o contenido con metadatos muy similares pero fracasa al intentar entender la "esencia" de una pelicula o serie. Las bajas puntuaciones de similitud obtenidas (generalmente por debajo de 0.15) confirman que el modelo no encuentra coincidencias sólidas, sino que agrupa títulos por palabras genéricas.

En el siguiente notebook se realizará una evaluación y un análisis mucho mayor para conocer de verdad el alcance y la capacidad recomendadora de este modelo.

REVIEW